In [13]:
import torch
from torch import nn
import torch.nn.functional as F
import torch.distributions as D
import numpy as np
import matplotlib.pyplot as plt
import collections

from models.cnp import *

In [14]:
def plot_functions(target_x, target_y, context_x, context_y, pred_y, var):
    """Plots the predicted mean and variance and the context points.

    Args: 
    target_x: An array of shape batchsize x number_targets x 1 that contains the
        x values of the target points.
    target_y: An array of shape batchsize x number_targets x 1 that contains the
        y values of the target points.
    context_x: An array of shape batchsize x number_context x 1 that contains 
        the x values of the context points.
    context_y: An array of shape batchsize x number_context x 1 that contains 
        the y values of the context points.
    pred_y: An array of shape batchsize x number_targets x 1  that contains the
        predicted means of the y values at the target points in target_x.
    pred_y: An array of shape batchsize x number_targets x 1  that contains the
        predicted variance of the y values at the target points in target_x.
    """
    # Plot everything
    plt.plot(target_x[0], pred_y[0], 'b', linewidth=2)
    plt.plot(target_x[0], target_y[0], 'k:', linewidth=2)
    plt.plot(context_x[0], context_y[0], 'ko', markersize=10)
    plt.fill_between(
      target_x[0, :, 0],
      pred_y[0, :, 0] - var[0, :, 0],
      pred_y[0, :, 0] + var[0, :, 0],
      alpha=0.2,
      facecolor='#65c9f7',
      interpolate=True)

    # Make the plot pretty
    plt.yticks([-2, 0, 2], fontsize=16)
    plt.xticks([-2, 0, 2], fontsize=16)
    plt.ylim([-2, 2])
    plt.grid('off')
    ax = plt.gca()
    #ax.set_axis_bgcolor('white')
    plt.show()

In [15]:
TRAINING_ITERATIONS = int(2e5)
MAX_CONTEXT_POINTS = 10
PLOT_AFTER = int(2e4)

# Sizes of the layers of the MLPs for the encoder and decoder
# The final output layer of the decoder outputs two values, one for the mean and
# one for the variance of the prediction at the target location
encoder_output_sizes = [128, 128, 128, 128]
decoder_output_sizes = [128, 128, 2]

# Define the model
encoder_input_size = 1 + 1 # x and y pairs are encoder into the context
decoder_input_size = 128 + 1 # target is concatenated onto the representation as input into the decoder
model = DeterministicModel(encoder_input_size, decoder_input_size)

# Set up the optimizer and train step
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for it in range(TRAINING_ITERATIONS):
    
    # Train dataset
    dataset_train = GPCurvesReader(
        batch_size=64, max_num_context=MAX_CONTEXT_POINTS)
    data_train = dataset_train.generate_curves()

    # Test dataset
    dataset_test = GPCurvesReader(
        batch_size=1, max_num_context=MAX_CONTEXT_POINTS, testing=True)
    data_test = dataset_test.generate_curves()
    
    log_prob, pred_y, var = model.forward(
            data_train.query, data_train.num_total_points,
            data_train.num_context_points, data_train.target_y
        )
    optimizer.zero_grad()
    loss = -torch.mean(log_prob)
    loss.backward()
    optimizer.step()

    # Plot the predictions in `PLOT_AFTER` intervals
    if it % PLOT_AFTER == 0:
        
        (context_x, context_y), target_x = data_test.query
        # Get the predicted mean and variance at the target points for the testing set
        log_prob, pred_y, var = model.forward(
            data_test.query, data_test.num_total_points,
            data_test.num_context_points, data_test.target_y)
        test_loss = -torch.mean(log_prob).detach().item()
        print('Iteration: {}, loss: {}'.format(it, test_loss))

        # Plot the prediction and the context
        plot_functions(target_x,
                       data_test.target_y,
                       context_x,
                       context_y,
                       pred_y.detach().numpy(),
                       var.detach().numpy())

RuntimeError: Expected size for first two dimensions of batch2 tensor to be: [1, 400] but got: [1, 401].